## **Instruction Fine-Tuning for LLMs**

**What is Instruction Fine-Tuning?**

After pre-training on massive text corpora, base LLMs are good at *completing text* but not at *following instructions*. Instruction fine-tuning bridges this gap by training the model on (instruction, input, response) triples so it learns to answer questions, follow commands, and hold conversations.

**Why does it matter?**
- Aligns model behaviour with user intent
- Improves zero-shot generalization to unseen tasks
- Turns a text-completion engine into an assistant

In [3]:
# Install required libraries (run once)
# !pip install datasets transformers torch

---
### **Loading a Dataset with Hugging Face `datasets`**

We'll use the `datasets` library to load a popular instruction-following dataset. Two common choices are:
- **yahma/alpaca-cleaned** – cleaned version of the original Stanford Alpaca dataset
- **tatsu-lab/alpaca** – the original Alpaca dataset

Each example contains: `instruction`, `input` (optional context), and `output`.

In [4]:
from datasets import load_dataset

# Load a small subset for demonstration
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:100]")
print(f"Number of examples: {len(dataset)}")
print(f"Features: {dataset.features}")
print()

# Inspect the first example
ex = dataset[0]
print("=== Example 0 ===")
print(f"Instruction: {ex['instruction']}")
print(f"Input:       {ex['input']}")
print(f"Output:      {ex['output']}")

Number of examples: 100
Features: {'output': Value('string'), 'input': Value('string'), 'instruction': Value('string')}

=== Example 0 ===
Instruction: Give three tips for staying healthy.
Input:       
Output:      1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


---
### **Alpaca Prompt Format**

Stanford's Alpaca paper used a simple template to format each example for training:

```
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
```

If there is no input (empty string), the `### Input:` line is often omitted.

**Why this format?**
- The **system prompt** sets the behavioural context.
- Clear **delimiters** (`### Instruction:`, `### Input:`, `### Response:`) help the model learn the structure.
- The model is trained to generate the response given the instruction + input prefix.

In [5]:
def format_alpaca(example):
    """Convert a dataset example into Alpaca-style prompt string."""
    prompt = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n"
    prompt += f"### Instruction:\n{example['instruction']}\n\n"
    if example['input']:
        prompt += f"### Input:\n{example['input']}\n\n"
    prompt += "### Response:\n"
    return prompt


# Format a few examples
for i in range(3):
    ex = dataset[i]
    prompt = format_alpaca(ex)
    print(f"===== Alpaca-formatted example {i} =====")
    print(prompt + ex['output'])
    print("=" * 50 + "\n")

===== Alpaca-formatted example 0 =====
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours

---
## **Hands-On Project: Data Batching for Instruction Fine-Tuning**

We now build a complete data batching pipeline:
1. Format all examples into Alpaca prompts
2. Tokenize them (input IDs + labels with -100 masking)
3. Create a PyTorch `Dataset` + `DataLoader` with dynamic padding
4. Inspect a real batch ready for training

In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

full_dataset = load_dataset("yahma/alpaca-cleaned", split="train")
print(f"Full dataset size: {len(full_dataset)}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

d:\anaconda3\envs\dsenv\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\anaconda3\envs\dsenv\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will b

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Full dataset size: 51760


In [7]:
def format_alpaca_prompt(example):
    prompt = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n"
    prompt += f"### Instruction:\n{example['instruction']}\n\n"
    if example['input']:
        prompt += f"### Input:\n{example['input']}\n\n"
    prompt += "### Response:\n"
    return prompt

prompts = [format_alpaca_prompt(ex) for ex in full_dataset]
responses = [ex['output'] for ex in full_dataset]

print("Prompt preview:\n", repr(prompts[0][:120]), "...")
print("Response preview:\n", repr(responses[0][:120]), "...")

Prompt preview:
 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that' ...
Response preview:
 '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean pr' ...


In [8]:
import torch

class InstructionDataset(torch.utils.data.Dataset):
    """PyTorch Dataset that tokenizes prompt+response and builds labels.
    Labels are set to -100 for prompt tokens so the loss is computed
    only on the response tokens (standard causal LM fine-tuning)."""
    def __init__(self, prompts, responses, tokenizer, max_length=512):
        self.prompts = prompts
        self.responses = responses
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        response = self.responses[idx]
        full_text = prompt + response

        enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            return_tensors=None,
        )
        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]

        prompt_len = len(self.tokenizer(prompt, return_tensors=None)["input_ids"])
        labels = [-100] * prompt_len + input_ids[prompt_len:]

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [9]:
def collate_fn(batch):
    """Dynamic padding: pad all sequences in the batch to the same length."""
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [10]:
dataset = InstructionDataset(prompts[:1000], responses[:1000], tokenizer)
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
)
print(f"Number of batches: {len(dataloader)}")

Number of batches: 250


In [12]:
batch = next(iter(dataloader))
print(f"input_ids shape:      {batch['input_ids'].shape}")       # (batch, seq_len)
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"labels shape:         {batch['labels'].shape}")
print()
print("=== Batch sample (example 0) ===")
print("input_ids:", batch['input_ids'][0].tolist()[:30], "...")
print("labels:   ", batch['labels'][0].tolist()[:30], "...")
print()
print("Decoded input:", tokenizer.decode(batch['input_ids'][0])[:200])
print("...")
print("Decoded labels (skipping -100):", tokenizer.decode(batch['labels'][0][batch['labels'][0] != -100])[:200])

input_ids shape:      torch.Size([4, 474])
attention_mask shape: torch.Size([4, 474])
labels shape:         torch.Size([4, 474])

=== Batch sample (example 0) ===
input_ids: [21106, 318, 281, 12064, 326, 8477, 257, 4876, 11, 20312, 351, 281, 5128, 326, 3769, 2252, 4732, 13, 19430, 257, 2882, 326, 20431, 32543, 262, 2581, 13, 198, 198, 21017] ...
labels:    [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100] ...

Decoded input: Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Generate a story about a
...
Decoded labels (skipping -100): Once upon a time, there was a young girl named Emily. Emily was a curious and adventurous girl with a love for science and all things unknown. One day, while exploring her local science museum, she st


---
### **What You Built**

| Component | Purpose |
|-----------|---------|
| `format_alpaca_prompt()` | Converts raw (instruction, input, output) into the Alpaca chat template |
| `InstructionDataset` | Tokenizes prompt+response, builds labels where prompt tokens are masked (`-100`) |
| `collate_fn` | Dynamically pads sequences in each batch to the longest sequence |
| `DataLoader` | Iterates over the dataset in shuffled batches ready for training |

**Key takeaway**: The model learns *only* from the response tokens because prompt positions are masked with `-100` in the labels. This is how instruction fine-tuning teaches a base model to follow instructions without penalising it for the prompt.